# Лабораторная работа 1.2 — эффект Комптона на графите

Датчик (сцинтилляционный спектрометр) поворачивают на угол $\theta$ вокруг графитовой мишени и для каждого угла фиксируют номер канала $N(\theta)$, отвечающий вершине фотопика. Проверяем
$$\frac{1}{N(\theta)} = \frac{1}{N(0)} + A(1-\cos\theta),$$
строя график $1/N(\theta)$ от $(1-\cos\theta)$: по пересечению прямой с осями находим $N_\text{наил}(0)$ и $N_\text{наил}(90)$, а из них — энергию покоя электрона $mc^2$.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from gostplot import GostPlot

In [ ]:
# Данные лабораторного практикума
angles_raw = np.array([x for x in range(0, 121, 10)])                                             # theta, град
channel_max = np.array([960, 960, 830, 786, 730, 672, 544, 498, 457, 415, 373, 352, 329],
                       dtype=float)                                                                # N(theta) — номер канала фотопика
assert len(angles_raw) == len(channel_max)

dtheta_deg = 1.0   # погрешность угла — половина деления лимба, град
dN = 3.2           # погрешность номера канала фотопика (полуширина пика); при наличии реальных
                   # полуширин под каждым углом замени на массив той же длины, что channel_max
E_gamma = 661.657  # кэВ — энергия гамма-квантов источника 137Cs
mc2_tab = 511.0    # кэВ — табличная энергия покоя электрона

In [ ]:
# Проверка формулы 1/N(θ) = 1/N(0) + A(1 - cosθ):
# по оси Y — 1/N(θ) для КАЖДОГО угла (включая θ=0), а не разность с N(0) —
# так наилучшее N(0) определяется всей прямой, а не одной сырой точкой

N_rev_val = 1 / channel_max
cos_val = 1 - np.cos(np.deg2rad(angles_raw))

dtheta = np.deg2rad(dtheta_deg)
dx = np.sin(np.deg2rad(angles_raw)) * dtheta      # перенос ошибки на (1 - cosθ)
dy = dN / channel_max**2                          # перенос ошибки на 1/N(θ)

plot1 = (GostPlot(cos_val, N_rev_val)
         .xlabel(r'$1-\cos\theta$')
         .ylabel(r'$1/N(\theta)$')
         .title('Эффект Комптона: $1/N(\\theta)$ от $(1-\\cos\\theta)$')
         .errors(dy=dy, dx=dx)
         .fit(1)
         .save2pdf('fig_1_2_compton.pdf'))
plot1.show()
print(plot1.result.report)
print(f'R² = {plot1.result.r2:.4f}')

In [ ]:
# Наилучшие N(0), N(90) — из пересечений прямой с осями,
# и энергия покоя электрона mc² = E_gamma * N(90) / (N(0) - N(90))

A, b = plot1.result.params      # y = A*x + b
cov = plot1.result.cov          # ковариация (A, b)

y0, y90 = b, A + b              # значения прямой в x=0 (θ=0) и x=1 (θ=90)
N0_best, N90_best = 1 / y0, 1 / y90

# ковариация y(0), y(90) через ковариацию параметров прямой: Var(y) = v^T Cov v
v0, v1 = np.array([0, 1]), np.array([1, 1])
var_y0, var_y90 = v0 @ cov @ v0, v1 @ cov @ v1
cov_y0_y90 = v0 @ cov @ v1

dN0  = np.sqrt(var_y0)  / y0**2
dN90 = np.sqrt(var_y90) / y90**2
cov_N0_N90 = cov_y0_y90 / (y0**2 * y90**2)

D = N0_best - N90_best
mc2 = E_gamma * N90_best / D

dmc2_dN0, dmc2_dN90 = -E_gamma * N90_best / D**2, E_gamma * N0_best / D**2
dmc2 = np.sqrt(dmc2_dN0**2 * dN0**2 + dmc2_dN90**2 * dN90**2
               + 2 * dmc2_dN0 * dmc2_dN90 * cov_N0_N90)

print(f'N_наил(0)  = {N0_best:.1f} ± {dN0:.1f}')
print(f'N_наил(90) = {N90_best:.1f} ± {dN90:.1f}')
print(f'mc² = {mc2:.1f} ± {dmc2:.1f} кэВ   (табл. 511 кэВ, откл. {(mc2/mc2_tab-1)*100:+.1f}%)')